In [ ]:
import os

from google.colab import files

# =========================================================
# STEP 1 — KAGGLE SETUP & DOWNLOAD
# =========================================================

print("=" * 60)
print("Upload your kaggle.json file from your local machine:")
print("=" * 60)
files.upload()

os.makedirs("/root/.config/kaggle", exist_ok=True)
os.system("mv kaggle.json /root/.config/kaggle/kaggle.json")
os.system("chmod 600 /root/.config/kaggle/kaggle.json")
os.system("pip install kaggle -q")

# ── PlantVillage ──────────────────────────────────────────
extract_path = "/content/PlantVillage"
os.makedirs(extract_path, exist_ok=True)
print("\nDownloading & extracting PlantVillage dataset from Kaggle...")
ret = os.system(
    f"kaggle datasets download -d emmarex/plantdisease --unzip -p {extract_path}"
)
if ret != 0:
    raise RuntimeError("PlantVillage download failed. Check your kaggle.json.")
print("PlantVillage downloaded.")

# ── PlantDoc ──────────────────────────────────────────────
pd_extract = "/content/PlantDoc"
os.makedirs(pd_extract, exist_ok=True)
print("\nDownloading & extracting PlantDoc dataset from Kaggle...")

PD_SLUGS = [
    "nirmalsankalana/plantdoc-dataset",   # primary (active as of 2025)
    "abdoelsayed2016/plantdoc",           # original (may be removed)
    "abdulhasibuddin/plant-doc-dataset",  # mirror
]
pd_ok = False
for slug in PD_SLUGS:
    print(f"  Trying slug: {slug}")
    ret = os.system(f"kaggle datasets download -d {slug} --unzip -p {pd_extract}")
    if ret == 0:
        print(f"  ✓ Downloaded with slug: {slug}")
        pd_ok = True
        break
    else:
        print(f"  ✗ Failed: {slug}")

if not pd_ok:
    raise RuntimeError(
        "All PlantDoc slugs failed. Search kaggle.com for 'PlantDoc' "
        "to find the current slug and update PD_SLUGS above."
    )
print("PlantDoc downloaded.")

# Print folder trees for debugging
print("\nPlantVillage folder structure:")
for root, dirs, _ in os.walk(extract_path):
    level = root.replace(extract_path, "").count(os.sep)
    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")

print("\nPlantDoc folder structure:")
for root, dirs, _ in os.walk(pd_extract):
    level = root.replace(pd_extract, "").count(os.sep)
    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")

# =========================================================
# STEP 2 — AUTO-DETECT PATHS (FIXED)
# =========================================================

def find_plantvillage_root(base, target_classes, max_depth=6):
    """
    Walk base and return the directory whose IMMEDIATE subfolders
    contain the most matches from target_classes.
    This is the PARENT of the class folders, not a class folder itself.
    """
    base = os.path.abspath(base)
    best_dir   = base
    best_count = 0
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        count = sum(1 for cls in target_classes if cls in dirs)
        if count > best_count:
            best_count = count
            best_dir   = root
    print(f"  -> {best_dir}  ({best_count}/{len(target_classes)} target classes found)")
    return best_dir


def find_split_dir(base, split_name, max_depth=6):
    """Find train or test folder by exact name match first, then substring."""
    base = os.path.abspath(base)
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        for d in dirs:
            if d.lower() == split_name.lower():
                return os.path.join(root, d)
    for root, dirs, _ in os.walk(base):
        depth = root.replace(base, "").count(os.sep)
        if depth > max_depth:
            dirs.clear()
            continue
        for d in dirs:
            if split_name.lower() in d.lower():
                return os.path.join(root, d)
    return None


# Target PV class folder names (single underscore, as in the dataset)
PV_TARGET_CLASSES = [
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_healthy",
]

PV_BASE = "/content/PlantVillage"
print("\nSearching for PlantVillage root (parent of class folders)...")
PV_DIR = find_plantvillage_root(PV_BASE, PV_TARGET_CLASSES)
print(f"PV_DIR: {PV_DIR}")

PD_BASE = "/content/PlantDoc"
print("\nSearching for PlantDoc train/test folders...")
PD_TRAIN = find_split_dir(PD_BASE, "train") or os.path.join(PD_BASE, "train")
PD_TEST  = find_split_dir(PD_BASE, "test")  or os.path.join(PD_BASE, "test")
print(f"PD_TRAIN: {PD_TRAIN}")
print(f"PD_TEST:  {PD_TEST}")

for label, path in [("PV_DIR", PV_DIR), ("PD_TRAIN", PD_TRAIN), ("PD_TEST", PD_TEST)]:
    if os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"  OK {label} exists  ({n} entries)")
    else:
        print(f"  MISSING {label}: {path}")

# =========================================================
# STEP 3 — IMPORTS
# =========================================================

import json
import random
import numpy as np
from collections import Counter

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import transforms
from torchvision.models import googlenet, GoogLeNet_Weights
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
)

# =========================================================
# STEP 4 — CONFIG
# =========================================================

SAVE_DIR    = "saved_models"
BATCH_SIZE  = 32
EPOCHS      = 25
LR          = 1e-4
FOLDS       = 5
SEED        = 42
PATIENCE    = 5
MIXUP_ALPHA = 0.1
GRAD_CLIP   = 1.0

# =========================================================
# STEP 5 — SEED & DEVICE
# =========================================================

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

os.makedirs(SAVE_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nUsing Device:", device)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# =========================================================
# STEP 6 — CLASS MAPPINGS
# =========================================================

SELECTED_CLASSES = [
    "Tomato_Bacterial_spot",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_healthy",
]

num_classes  = len(SELECTED_CLASSES)
new_to_class = {i: cls for i, cls in enumerate(SELECTED_CLASSES)}
class_to_idx = {cls: i for i, cls in enumerate(SELECTED_CLASSES)}

# PlantVillage: single-underscore folder names
PV_MAP = {cls: idx for cls, idx in class_to_idx.items()}

# PlantDoc folder name -> label mapping.
# Covers actual folder names seen in nirmalsankalana/plantdoc-dataset
# plus alternate naming from other slugs.
PD_MAP = {
    # -- nirmalsankalana slug (confirmed from logs) --
    "Tomato_Early_blight_leaf":   class_to_idx["Tomato_Early_blight"],
    "Tomato_leaf_bacterial_spot": class_to_idx["Tomato_Bacterial_spot"],
    "Tomato_leaf_late_blight":    class_to_idx["Tomato_Late_blight"],
    "Tomato_mold_leaf":           class_to_idx["Tomato_Leaf_Mold"],
    "Tomato_leaf":                class_to_idx["Tomato_healthy"],
    # -- space + title-case (other slugs) --
    "Tomato Bacterial Spot":      class_to_idx["Tomato_Bacterial_spot"],
    "Tomato Early Blight":        class_to_idx["Tomato_Early_blight"],
    "Tomato Late Blight":         class_to_idx["Tomato_Late_blight"],
    "Tomato Leaf Mold":           class_to_idx["Tomato_Leaf_Mold"],
    "Tomato healthy":             class_to_idx["Tomato_healthy"],
    "Tomato Healthy":             class_to_idx["Tomato_healthy"],
    # -- double-underscore variants --
    "Tomato__Bacterial_spot":     class_to_idx["Tomato_Bacterial_spot"],
    "Tomato__Early_blight":       class_to_idx["Tomato_Early_blight"],
    "Tomato__Late_blight":        class_to_idx["Tomato_Late_blight"],
    "Tomato__Leaf_Mold":          class_to_idx["Tomato_Leaf_Mold"],
    "Tomato__healthy":            class_to_idx["Tomato_healthy"],
    # -- single-underscore fallback --
    "Tomato_Bacterial_spot":      class_to_idx["Tomato_Bacterial_spot"],
    "Tomato_Early_blight":        class_to_idx["Tomato_Early_blight"],
    "Tomato_Late_blight":         class_to_idx["Tomato_Late_blight"],
    "Tomato_Leaf_Mold":           class_to_idx["Tomato_Leaf_Mold"],
    "Tomato_healthy":             class_to_idx["Tomato_healthy"],
}

# =========================================================
# STEP 7 — TRANSFORMS
# =========================================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomAdjustSharpness(sharpness_factor=2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# =========================================================
# STEP 8 — SAMPLE COLLECTOR
# =========================================================

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def collect_samples(base_dir, mapping):
    """
    For each subfolder of base_dir that matches a key in mapping,
    collect (image_path, label_index) pairs.
    """
    if not os.path.isdir(base_dir):
        print(f"  [SKIP] Directory not found: {base_dir}")
        return []

    samples       = []
    found_folders = set(os.listdir(base_dir))
    seen_paths    = set()   # avoid double-counting same physical folder

    for folder, label_idx in mapping.items():
        if folder not in found_folders:
            continue
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue
        real_path = os.path.realpath(folder_path)
        if real_path in seen_paths:
            continue
        seen_paths.add(real_path)

        count = 0
        for fname in os.listdir(folder_path):
            if fname.lower().endswith(IMG_EXTS):
                samples.append((os.path.join(folder_path, fname), label_idx))
                count += 1
        print(f"  {folder:<45} {count:>5} images -> class {label_idx} ({new_to_class[label_idx]})")

    unmatched = [f for f in found_folders
                 if os.path.isdir(os.path.join(base_dir, f)) and f not in mapping]
    if unmatched:
        print(f"  [unmatched, not in mapping]: {unmatched}")

    return samples


print("\n-- PlantVillage samples --")
pv_all   = collect_samples(PV_DIR,   PV_MAP)

print("\n-- PlantDoc train samples --")
pd_train = collect_samples(PD_TRAIN, PD_MAP)

print("\n-- PlantDoc test samples --")
pd_test  = collect_samples(PD_TEST,  PD_MAP)

print(f"\nPlantVillage  : {len(pv_all):>6} images")
print(f"PlantDoc train: {len(pd_train):>6} images")
print(f"PlantDoc test : {len(pd_test):>6} images")

if len(pv_all) == 0:
    raise RuntimeError(
        f"No PlantVillage images found!\n"
        f"PV_DIR = {PV_DIR}\n"
        f"Folders present: {os.listdir(PV_DIR)}\n"
        f"PV_MAP keys: {list(PV_MAP.keys())}"
    )

# =========================================================
# STEP 9 — COMBINED DATASET
# =========================================================

class SampleListDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception:
            image = Image.new("RGB", (224, 224))
        if self.transform:
            image = self.transform(image)
        return image, label


all_train_samples = pv_all + pd_train
all_test_samples  = pd_test

print(f"\nTotal train pool : {len(all_train_samples)}")
print(f"Total test set   : {len(all_test_samples)}")

print("\nClass distribution in train pool:")
cnt = Counter(label for _, label in all_train_samples)
for idx in range(num_classes):
    print(f"  [{idx}] {new_to_class[idx]:<35} {cnt[idx]:>5}")

# =========================================================
# STEP 10 — REMAPPED SUBSET (for K-Fold)
# =========================================================

class RemappedSubset(Dataset):
    def __init__(self, dataset, positions):
        self.dataset   = dataset
        self.positions = positions

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, idx):
        return self.dataset[self.positions[idx]]

# =========================================================
# STEP 11 — MIXUP
# =========================================================

def mixup_data(x, y, alpha=0.1):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(device)
    return lam * x + (1 - lam) * x[index], y, y[index], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# =========================================================
# STEP 12 — COORDINATE ATTENTION
# =========================================================

class CoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super().__init__()
        mid         = max(8, in_channels // reduction)
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        self.conv1  = nn.Conv2d(in_channels, mid, kernel_size=1, bias=False)
        self.bn1    = nn.BatchNorm2d(mid)
        self.act    = nn.Hardswish()
        self.conv_h = nn.Conv2d(mid, in_channels, kernel_size=1, bias=False)
        self.conv_w = nn.Conv2d(mid, in_channels, kernel_size=1, bias=False)

    def forward(self, x):
        B, C, H, W = x.shape
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y   = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h_, x_w_ = torch.split(y, [H, W], dim=2)
        a_h = torch.sigmoid(self.conv_h(x_h_))
        a_w = torch.sigmoid(self.conv_w(x_w_.permute(0, 1, 3, 2)))
        return x * a_h * a_w

# =========================================================
# STEP 13 — SE BLOCK
# =========================================================

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1)

# =========================================================
# STEP 14 — FOCAL LOSS
# =========================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2, smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(label_smoothing=smoothing)

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt      = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

# =========================================================
# STEP 15 — MODEL
# =========================================================

class ModifiedGoogLeNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        base            = googlenet(weights=GoogLeNet_Weights.DEFAULT)
        base.aux_logits = False
        base.aux1       = None
        base.aux2       = None

        self.phase1 = nn.Sequential(
            base.conv1, base.maxpool1, base.conv2, base.conv3, base.maxpool2)
        self.phase2 = nn.Sequential(
            base.inception3a, base.inception3b, base.maxpool3)
        self.phase3 = nn.Sequential(
            base.inception4a, base.inception4b, base.inception4c,
            base.inception4d, base.inception4e, base.maxpool4)
        self.phase4 = nn.Sequential(base.inception5a, base.inception5b)

        self.se12 = SEBlock(192)
        self.se23 = SEBlock(480)
        self.se34 = SEBlock(832)
        self.ca12 = CoordinateAttention(192)
        self.ca34 = CoordinateAttention(832)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.drop = nn.Dropout(0.5)
        self.fc   = nn.Linear(1024, num_classes)

    def forward(self, x):
        x = self.phase1(x)
        x = self.se12(x) + x
        x = self.ca12(x)
        x = self.phase2(x)
        x = self.se23(x) + x
        x = self.phase3(x)
        x = self.se34(x) + x
        x = self.ca34(x)
        x = self.phase4(x)
        return self.fc(self.drop(torch.flatten(self.pool(x), 1)))

# =========================================================
# STEP 16 — METRICS
# =========================================================

def compute_metrics(labels, preds, probs):
    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="macro", zero_division=0)
    rec  = recall_score(labels, preds, average="macro", zero_division=0)
    f1   = f1_score(labels, preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr")
    except Exception:
        auc = float("nan")
    return acc, prec, rec, f1, auc


def print_per_class_accuracy(split_name, labels, preds):
    print(f"\n{split_name} Per-Class Accuracy:")
    for c in range(num_classes):
        mask    = labels == c
        total   = mask.sum()
        correct = (preds[mask] == c).sum() if total > 0 else 0
        acc_c   = (correct / total) if total > 0 else 0.0
        print(f"  [{c}] {new_to_class[c]:<35} {correct}/{total}  acc={acc_c:.4f}")


def print_metrics(split, acc, prec, rec, f1, auc=None, loss=None):
    auc_str  = f"{auc:.4f}" if auc is not None and not np.isnan(auc) else "N/A"
    loss_str = f"Loss:{loss:.4f}  " if loss is not None else ""
    print(f"{split:<6}  {loss_str}Acc:{acc:.4f}  Prec:{prec:.4f}  "
          f"Rec:{rec:.4f}  F1:{f1:.4f}  AUC:{auc_str}")

# =========================================================
# STEP 17 — TRAIN ONE EPOCH
# =========================================================

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss                       = 0.0
    all_preds, all_labels, all_probs = [], [], []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)
        images, labels_a, labels_b, lam = mixup_data(images, labels, MIXUP_ALPHA)

        optimizer.zero_grad()
        outputs = model(images)
        loss    = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        probs       = F.softmax(outputs.detach(), dim=1)
        predicted   = torch.argmax(probs, dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, np.array(all_preds), np.array(all_labels), np.array(all_probs)

# =========================================================
# STEP 18 — EVALUATE
# =========================================================

def evaluate(model, loader, criterion=None):
    model.eval()
    total_loss                       = 0.0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            labels  = labels.to(device)
            outputs = model(images)
            if criterion is not None:
                total_loss += criterion(outputs, labels).item() * images.size(0)
            probs     = F.softmax(outputs, dim=1)
            predicted = torch.argmax(probs, dim=1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset) if criterion is not None else None
    return avg_loss, np.array(all_preds), np.array(all_labels), np.array(all_probs)

# =========================================================
# STEP 19 — PREPARE ARRAYS FOR K-FOLD
# =========================================================

full_dataset_train = SampleListDataset(all_train_samples, transform=train_transform)
full_dataset_val   = SampleListDataset(all_train_samples, transform=val_transform)

selected_positions = np.arange(len(all_train_samples))
selected_labels    = np.array([label for _, label in all_train_samples])

print("\nSelected Classes:")
for i, cls in new_to_class.items():
    print(f"  {i} -> {cls}")
print(f"\nTotal Samples for CV: {len(selected_positions)}")

# =========================================================
# STEP 20 — K-FOLD CROSS VALIDATION TRAINING
# =========================================================

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

best_global_f1   = 0.0
best_global_ckpt = None
best_fold        = -1

for fold, (tr_idx, val_idx) in enumerate(
    skf.split(selected_positions, selected_labels)
):
    print("\n" + "=" * 70)
    print(f"FOLD {fold + 1}/{FOLDS}")
    print("=" * 70)

    train_subset = RemappedSubset(full_dataset_train, selected_positions[tr_idx])
    val_subset   = RemappedSubset(full_dataset_val,   selected_positions[val_idx])

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    model     = ModifiedGoogLeNet(num_classes).to(device)
    criterion = FocalLoss(gamma=2, smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_f1            = 0.0
    best_ckpt              = None
    epochs_without_improve = 0

    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch + 1}/{EPOCHS}")

        train_loss, tr_preds, tr_labels, tr_probs = train_one_epoch(
            model, train_loader, criterion, optimizer)
        train_m = compute_metrics(tr_labels, tr_preds, tr_probs)
        print_metrics("Train", *train_m, loss=train_loss)
        print_per_class_accuracy("Train", tr_labels, tr_preds)

        val_loss, val_preds, val_labels, val_probs = evaluate(
            model, val_loader, criterion)
        val_m = compute_metrics(val_labels, val_preds, val_probs)
        print_metrics("Val", *val_m, loss=val_loss)
        print_per_class_accuracy("Val", val_labels, val_preds)

        scheduler.step()

        if val_m[3] > best_val_f1:
            best_val_f1            = val_m[3]
            best_ckpt              = {k: v.cpu().clone()
                                      for k, v in model.state_dict().items()}
            epochs_without_improve = 0
            print(f"\n  New Best Val F1: {best_val_f1:.4f}")
        else:
            epochs_without_improve += 1
            print(f"  EarlyStopping Counter: {epochs_without_improve}/{PATIENCE}")
            if epochs_without_improve >= PATIENCE:
                print("\n  Early stopping triggered.")
                break

    # Fold final validation
    model.load_state_dict(best_ckpt)
    _, val_preds, val_labels, val_probs = evaluate(model, val_loader)
    val_m_best = compute_metrics(val_labels, val_preds, val_probs)

    print("\n" + "=" * 70)
    print(f"Fold {fold + 1} Best Validation Results")
    print("=" * 70)
    print_metrics("Val", *val_m_best)
    print_per_class_accuracy("Val", val_labels, val_preds)
    print("\n", classification_report(
        val_labels, val_preds,
        target_names=[new_to_class[i] for i in range(num_classes)],
        digits=4, zero_division=0,
    ))

    fold_path = os.path.join(SAVE_DIR, f"fold_{fold + 1}_best.pth")
    torch.save({
        "fold":        fold + 1,
        "num_classes": num_classes,
        "class_names": [new_to_class[i] for i in range(num_classes)],
        "val_f1":      best_val_f1,
        "model_state": best_ckpt,
    }, fold_path)
    print(f"\nSaved: {fold_path}")

    if best_val_f1 > best_global_f1:
        best_global_f1   = best_val_f1
        best_global_ckpt = best_ckpt
        best_fold        = fold + 1

# =========================================================
# STEP 21 — FINAL TEST-SET EVALUATION (PlantDoc test)
# =========================================================

if len(all_test_samples) > 0:
    print("\n" + "=" * 70)
    print("FINAL TEST SET EVALUATION  (PlantDoc held-out test)")
    print("=" * 70)

    test_dataset = SampleListDataset(all_test_samples, transform=val_transform)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=4, pin_memory=True)

    best_model = ModifiedGoogLeNet(num_classes).to(device)
    best_model.load_state_dict(best_global_ckpt)

    _, test_preds, test_labels, test_probs = evaluate(best_model, test_loader)
    test_m = compute_metrics(test_labels, test_preds, test_probs)
    print_metrics("Test", *test_m)
    print_per_class_accuracy("Test", test_labels, test_preds)
    print("\n", classification_report(
        test_labels, test_preds,
        target_names=[new_to_class[i] for i in range(num_classes)],
        digits=4, zero_division=0,
    ))
else:
    print("\n[INFO] No PlantDoc test samples found — skipping test evaluation.")

# =========================================================
# STEP 22 — SAVE FINAL BEST MODEL
# =========================================================

final_model_path = os.path.join(SAVE_DIR, "best_model_final.pth")
class_names      = [new_to_class[i] for i in range(num_classes)]

torch.save({
    "fold":        best_fold,
    "num_classes": num_classes,
    "class_names": class_names,
    "val_f1":      best_global_f1,
    "model_state": best_global_ckpt,
}, final_model_path)

with open(os.path.join(SAVE_DIR, "class_names.json"), "w") as f:
    json.dump(class_names, f, indent=2)

print("\n" + "=" * 70)
print(f"FINAL MODEL SAVED : {final_model_path}")
print(f"BEST FOLD         : {best_fold}")
print(f"BEST VAL F1       : {best_global_f1:.4f}")
print("=" * 70)

Upload your kaggle.json file from your local machine:


Saving kaggle.json to kaggle.json

PlantVillage downloaded.

  Trying slug: nirmalsankalana/plantdoc-dataset
  ✓ Downloaded with slug: nirmalsankalana/plantdoc-dataset
PlantDoc downloaded.

PlantVillage folder structure:
PlantVillage/
  plantvillage/
    PlantVillage/
      Tomato_Bacterial_spot/
      Pepper__bell___Bacterial_spot/
      Tomato_Leaf_Mold/
      Pepper__bell___healthy/
      Tomato_Spider_mites_Two_spotted_spider_mite/
      Tomato__Target_Spot/
      Tomato_healthy/
      Tomato_Early_blight/
      Tomato__Tomato_YellowLeaf__Curl_Virus/
      Potato___healthy/
      Tomato_Septoria_leaf_spot/
      Tomato_Late_blight/
      Potato___Late_blight/
      Potato___Early_blight/
      Tomato__Tomato_mosaic_virus/
  PlantVillage/
    Tomato_Bacterial_spot/
    Pepper__bell___Bacterial_spot/
    Tomato_Leaf_Mold/
    Pepper__bell___healthy/
    Tomato_Spider_mites_Two_spotted_spider_mite/
    Tomato__Target_Spot/
    Tomato_healthy/
    Tomato_Early_blight/
    Tomato__Tomat

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Downloading: "https://download.pytorch.org/models/googlenet-1378be20.pth" to /root/.cache/torch/hub/checkpoints/googlenet-1378be20.pth


100%|██████████| 49.7M/49.7M [00:00<00:00, 138MB/s]


...

Val     Loss:0.0716  Acc:0.9812  Prec:0.9807  Rec:0.9782  F1:0.9794  AUC:0.9988

Val Per-Class Accuracy:
  [0] Tomato_Bacterial_spot               437/445  acc=0.9820
  [1] Tomato_Early_blight                 209/216  acc=0.9676
  [2] Tomato_Late_blight                  396/402  acc=0.9851
  [3] Tomato_Leaf_Mold                    198/207  acc=0.9565
  [4] Tomato_healthy                      327/327  acc=1.0000
  EarlyStopping Counter: 1/5


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



Fold 5 Best Validation Results
Val     Acc:0.9818  Prec:0.9807  Rec:0.9792  F1:0.9799  AUC:0.9987

Val Per-Class Accuracy:
  [0] Tomato_Bacterial_spot               437/445  acc=0.9820
  [1] Tomato_Early_blight                 210/216  acc=0.9722
  [2] Tomato_Late_blight                  396/402  acc=0.9851
  [3] Tomato_Leaf_Mold                    198/207  acc=0.9565
  [4] Tomato_healthy                      327/327  acc=1.0000

                        precision    recall  f1-score   support

Tomato_Bacterial_spot     0.9776    0.9820    0.9798       445
  Tomato_Early_blight     0.9677    0.9722    0.9700       216
   Tomato_Late_blight     0.9900    0.9851    0.9875       402
     Tomato_Leaf_Mold     0.9802    0.9565    0.9682       207
       Tomato_healthy     0.9879    1.0000    0.9939       327

             accuracy                         0.9818      1597
            macro avg     0.9807    0.9792    0.9799      1597
         weighted avg     0.9818    0.9818    0.9818      

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Test    Acc:0.5000  Prec:0.6467  Rec:0.4733  F1:0.4758  AUC:0.8612

Test Per-Class Accuracy:
  [0] Tomato_Bacterial_spot               7/9  acc=0.7778
  [1] Tomato_Early_blight                 2/9  acc=0.2222
  [2] Tomato_Late_blight                  7/10  acc=0.7000
  [3] Tomato_Leaf_Mold                    1/6  acc=0.1667
  [4] Tomato_healthy                      4/8  acc=0.5000

                        precision    recall  f1-score   support

Tomato_Bacterial_spot     0.3333    0.7778    0.4667         9
  Tomato_Early_blight     1.0000    0.2222    0.3636         9
   Tomato_Late_blight     0.7000    0.7000    0.7000        10
     Tomato_Leaf_Mold     0.2000    0.1667    0.1818         6
       Tomato_healthy     1.0000    0.5000    0.6667         8

             accuracy                         0.5000        42
            macro avg     0.6467    0.4733    0.4758        42
         weighted avg     0.6714    0.5000    0.4975        42


FINAL MODEL SAVED : saved_models/best_model

In [6]:
# =========================================================
# COMPLETE TESTING CODE
# FOR MODIFIED GOOGLENET TOMATO DISEASE MODEL
#
# TESTS ON:
# ✅ PlantDoc TEST SET
# ✅ PlantVillage TEST SET (optional)
#
# PRINTS:
# ✅ Overall Accuracy
# ✅ Precision
# ✅ Recall
# ✅ F1 Score
# ✅ AUC
# ✅ Class-wise Accuracy
# ✅ Classification Report
# ✅ Confusion Matrix
# =========================================================

import os
import json
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import transforms
from torchvision.models import googlenet, GoogLeNet_Weights

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# =========================================================
# CONFIG
# =========================================================

MODEL_PATH = "/content/saved_models/best_model_final.pth"

BATCH_SIZE = 32

IMG_SIZE = 224

NUM_WORKERS = 2

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using Device:", device)

# =========================================================
# CLASS NAMES
# =========================================================

CLASS_NAMES = [

    "Tomato_Bacterial_spot",

    "Tomato_Early_blight",

    "Tomato_Late_blight",

    "Tomato_Leaf_Mold",

    "Tomato_healthy"
]

CLASS_TO_IDX = {

    c:i

    for i,c in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {

    i:c

    for c,i in CLASS_TO_IDX.items()
}

NUM_CLASSES = len(CLASS_NAMES)

# =========================================================
# TRANSFORMS
# =========================================================

transform = transforms.Compose([

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

# =========================================================
# PLANTDOC MAPPING
# =========================================================

PD_MAP = {

    "Tomato_Early_blight_leaf":
        CLASS_TO_IDX["Tomato_Early_blight"],

    "Tomato_leaf_bacterial_spot":
        CLASS_TO_IDX["Tomato_Bacterial_spot"],

    "Tomato_leaf_late_blight":
        CLASS_TO_IDX["Tomato_Late_blight"],

    "Tomato_mold_leaf":
        CLASS_TO_IDX["Tomato_Leaf_Mold"],

    "Tomato_leaf":
        CLASS_TO_IDX["Tomato_healthy"],
}

# =========================================================
# PLANTVILLAGE MAPPING
# =========================================================

PV_MAP = {

    "Tomato_Bacterial_spot":
        CLASS_TO_IDX["Tomato_Bacterial_spot"],

    "Tomato_Early_blight":
        CLASS_TO_IDX["Tomato_Early_blight"],

    "Tomato_Late_blight":
        CLASS_TO_IDX["Tomato_Late_blight"],

    "Tomato_Leaf_Mold":
        CLASS_TO_IDX["Tomato_Leaf_Mold"],

    "Tomato_healthy":
        CLASS_TO_IDX["Tomato_healthy"],
}

# =========================================================
# DATASET PATHS
# =========================================================

PD_TEST_DIR = "/content/PlantDoc/PlantDoc-Dataset/test"

PV_DIR = "/content/PlantVillage/plantvillage dataset/color"

# =========================================================
# DATASET
# =========================================================

class SampleDataset(Dataset):

    def __init__(self, samples, transform=None):

        self.samples = samples

        self.transform = transform

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        try:

            image = Image.open(img_path).convert("RGB")

        except:

            image = Image.new(
                "RGB",
                (IMG_SIZE, IMG_SIZE)
            )

        if self.transform:

            image = self.transform(image)

        return image, label

# =========================================================
# COLLECT SAMPLES
# =========================================================

IMG_EXTS = (

    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
)

def collect_samples(base_dir, mapping):

    samples = []

    if not os.path.isdir(base_dir):

        return samples

    for folder, label in mapping.items():

        folder_path = os.path.join(base_dir, folder)

        if not os.path.isdir(folder_path):

            continue

        count = 0

        for fname in os.listdir(folder_path):

            if fname.lower().endswith(IMG_EXTS):

                samples.append(

                    (
                        os.path.join(folder_path, fname),
                        label
                    )
                )

                count += 1

        print(
            f"{folder:<40} "
            f"{count:>5} images"
        )

    return samples

# =========================================================
# COORDINATE ATTENTION
# =========================================================

class CoordinateAttention(nn.Module):

    def __init__(
        self,
        in_channels,
        reduction=32
    ):

        super().__init__()

        mid = max(
            8,
            in_channels // reduction
        )

        self.pool_h = nn.AdaptiveAvgPool2d((None,1))

        self.pool_w = nn.AdaptiveAvgPool2d((1,None))

        self.conv1 = nn.Conv2d(
            in_channels,
            mid,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(mid)

        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(
            mid,
            in_channels,
            kernel_size=1,
            bias=False
        )

        self.conv_w = nn.Conv2d(
            mid,
            in_channels,
            kernel_size=1,
            bias=False
        )

    def forward(self, x):

        B,C,H,W = x.shape

        x_h = self.pool_h(x)

        x_w = self.pool_w(x).permute(0,1,3,2)

        y = torch.cat([x_h, x_w], dim=2)

        y = self.act(self.bn1(self.conv1(y)))

        x_h, x_w = torch.split(y, [H,W], dim=2)

        x_w = x_w.permute(0,1,3,2)

        a_h = torch.sigmoid(self.conv_h(x_h))

        a_w = torch.sigmoid(self.conv_w(x_w))

        return x * a_h * a_w

# =========================================================
# SE BLOCK
# =========================================================

class SEBlock(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(

            nn.Linear(
                channels,
                channels // reduction
            ),

            nn.ReLU(),

            nn.Linear(
                channels // reduction,
                channels
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        b,c,_,_ = x.size()

        y = self.pool(x).view(b,c)

        y = self.fc(y).view(b,c,1,1)

        return x * y

# =========================================================
# MODEL
# =========================================================

class ModifiedGoogLeNet(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        base = googlenet(
            weights=GoogLeNet_Weights.DEFAULT
        )

        base.aux_logits = False

        base.aux1 = None

        base.aux2 = None

        self.phase1 = nn.Sequential(

            base.conv1,

            base.maxpool1,

            base.conv2,

            base.conv3,

            base.maxpool2
        )

        self.phase2 = nn.Sequential(

            base.inception3a,

            base.inception3b,

            base.maxpool3
        )

        self.phase3 = nn.Sequential(

            base.inception4a,

            base.inception4b,

            base.inception4c,

            base.inception4d,

            base.inception4e,

            base.maxpool4
        )

        self.phase4 = nn.Sequential(

            base.inception5a,

            base.inception5b
        )

        self.se12 = SEBlock(192)

        self.se23 = SEBlock(480)

        self.se34 = SEBlock(832)

        self.ca12 = CoordinateAttention(192)

        self.ca34 = CoordinateAttention(832)

        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.drop = nn.Dropout(0.5)

        self.fc = nn.Linear(
            1024,
            num_classes
        )

    def forward(self, x):

        x = self.phase1(x)

        x = self.se12(x) + x

        x = self.ca12(x)

        x = self.phase2(x)

        x = self.se23(x) + x

        x = self.phase3(x)

        x = self.se34(x) + x

        x = self.ca34(x)

        x = self.phase4(x)

        x = self.pool(x)

        x = torch.flatten(x,1)

        x = self.drop(x)

        x = self.fc(x)

        return x

# =========================================================
# METRICS
# =========================================================

def compute_metrics(labels, preds, probs):

    acc = accuracy_score(labels, preds)

    prec = precision_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    rec = recall_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        labels,
        preds,
        average="macro",
        zero_division=0
    )

    try:

        auc = roc_auc_score(
            labels,
            probs,
            multi_class="ovr"
        )

    except:

        auc = float("nan")

    return acc, prec, rec, f1, auc

# =========================================================
# EVALUATE
# =========================================================

def evaluate(model, loader):

    model.eval()

    all_preds = []

    all_labels = []

    all_probs = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            outputs = model(images)

            probs = F.softmax(outputs, dim=1)

            preds = torch.argmax(probs, dim=1)

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_labels.extend(
                labels.numpy()
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

    return (

        np.array(all_preds),

        np.array(all_labels),

        np.array(all_probs)
    )

# =========================================================
# LOAD MODEL
# =========================================================

print("\nLoading Model...")

checkpoint = torch.load(
    MODEL_PATH,
    map_location=device
)

model = ModifiedGoogLeNet(NUM_CLASSES)

model.load_state_dict(
    checkpoint["model_state"]
)

model = model.to(device)

print("Model Loaded Successfully")

# =========================================================
# TEST ON PLANTDOC
# =========================================================

print("\n" + "="*70)

print("PLANTDOC TEST SET")

print("="*70)

pd_test_samples = collect_samples(
    PD_TEST_DIR,
    PD_MAP
)

print(
    "\nTotal PlantDoc Test Images:",
    len(pd_test_samples)
)

pd_dataset = SampleDataset(
    pd_test_samples,
    transform
)

pd_loader = DataLoader(

    pd_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS
)

pd_preds, pd_labels, pd_probs = evaluate(
    model,
    pd_loader
)

pd_metrics = compute_metrics(
    pd_labels,
    pd_preds,
    pd_probs
)

print("\nOverall Metrics")

print(f"Accuracy  : {pd_metrics[0]:.4f}")

print(f"Precision : {pd_metrics[1]:.4f}")

print(f"Recall    : {pd_metrics[2]:.4f}")

print(f"F1 Score  : {pd_metrics[3]:.4f}")

print(f"AUC       : {pd_metrics[4]:.4f}")

print("\nPer-Class Accuracy")

for c in range(NUM_CLASSES):

    mask = pd_labels == c

    total = mask.sum()

    correct = (
        pd_preds[mask] == c
    ).sum()

    acc = (
        correct / total
        if total > 0
        else 0
    )

    print(

        f"{IDX_TO_CLASS[c]:<35} "

        f"{correct}/{total} "

        f"acc={acc:.4f}"
    )

print("\nClassification Report\n")

print(

    classification_report(

        pd_labels,

        pd_preds,

        target_names=CLASS_NAMES,

        digits=4,

        zero_division=0
    )
)

print("\nConfusion Matrix\n")

print(

    confusion_matrix(
        pd_labels,
        pd_preds
    )
)

# =========================================================
# OPTIONAL — TEST ON PLANTVILLAGE
# =========================================================

print("\n" + "="*70)

print("PLANTVILLAGE TEST")

print("="*70)

pv_samples = collect_samples(
    PV_DIR,
    PV_MAP
)

print(
    "\nTotal PlantVillage Images:",
    len(pv_samples)
)

pv_dataset = SampleDataset(
    pv_samples,
    transform
)

pv_loader = DataLoader(

    pv_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS
)

pv_preds, pv_labels, pv_probs = evaluate(
    model,
    pv_loader
)

pv_metrics = compute_metrics(
    pv_labels,
    pv_preds,
    pv_probs
)

print("\nOverall Metrics")

print(f"Accuracy  : {pv_metrics[0]:.4f}")

print(f"Precision : {pv_metrics[1]:.4f}")

print(f"Recall    : {pv_metrics[2]:.4f}")

print(f"F1 Score  : {pv_metrics[3]:.4f}")

print(f"AUC       : {pv_metrics[4]:.4f}")

print("\nPer-Class Accuracy")

for c in range(NUM_CLASSES):

    mask = pv_labels == c

    total = mask.sum()

    correct = (
        pv_preds[mask] == c
    ).sum()

    acc = (
        correct / total
        if total > 0
        else 0
    )

    print(

        f"{IDX_TO_CLASS[c]:<35} "

        f"{correct}/{total} "

        f"acc={acc:.4f}"
    )

print("\nClassification Report\n")

print(

    classification_report(

        pv_labels,

        pv_preds,

        target_names=CLASS_NAMES,

        digits=4,

        zero_division=0
    )
)

print("\nConfusion Matrix\n")

print(

    confusion_matrix(
        pv_labels,
        pv_preds
    )
)

Using Device: cuda

Loading Model...
Model Loaded Successfully

PLANTDOC TEST SET

Total PlantDoc Test Images: 0

Overall Metrics
Accuracy  : nan
Precision : nan
Recall    : nan
F1 Score  : nan
AUC       : nan

Per-Class Accuracy
Tomato_Bacterial_spot               0/0 acc=0.0000
Tomato_Early_blight                 0/0 acc=0.0000
Tomato_Late_blight                  0/0 acc=0.0000
Tomato_Leaf_Mold                    0/0 acc=0.0000
Tomato_healthy                      0/0 acc=0.0000

Classification Report



/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


ValueError: Number of classes, 0, does not match size of target_names, 5. Try specifying the labels parameter